In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

In [ ]:
open_router_api_key = os.getenv("OPEN_ROUTER_API_KEY")

if not open_router_api_key:
    raise ValueError("OPEN_ROUTER_API_KEY environment variable is not set")
else:
    print("OPEN_ROUTER_API_KEY environment variable is set")

In [ ]:
open_router = OpenAI(api_key=open_router_api_key, base_url="https://openrouter.ai/api/v1")

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant."
USER_PROMPT = "Tell me about virat kohli"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

In [ ]:
def call_llm(messages):
    stream = open_router.chat.completions.create(
        model="openrouter/free",
        messages=messages,
        stream=True
    )
    response = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)
            response += chunk.choices[0].delta.content
    messages.append({"role": "assistant", "content": response})

In [ ]:
messages.append(
    {"role": "user", "content": "why we can call him as king?"}
)
call_llm(messages)

In [ ]:
print(messages)

In [ ]:
messages.append(
    {"role": "user", "content": "What is the name of his fiance?"}
)
call_llm(messages)

# fetch the conversation from the database
# if the conversation is not found, create a new one
# if the conversation is found, use the existing conversation

In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")

db = client["llm_chat"]
conversations = db["conversations"]


In [ ]:
import uuid
def call_llm(messages: list[dict], userId: str):
    stream = open_router.chat.completions.create(
            model="openrouter/free",
            messages=messages,
            stream=True
        )
    response = ''
    for chunk in stream:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)
            response += chunk.choices[0].delta.content
    conversation = conversations.find_one({"userId": userId})
    messages.append({"role": "assistant", "content": response})
    if not conversation:
        conversation_id = str(uuid.uuid4())
        conversations.insert_one({"userId": userId, "conversationId": conversation_id, "messages": messages})
    else:
        conversation_id = conversation["conversationId"]
        conversation_messages = conversation["messages"]
        conversation_messages.append({"role": "user", "content": messages[-2]["content"]})
        conversation_messages.append({"role": "assistant", "content": messages[-1]["content"]})
        conversations.update_one({"userId": userId, "conversationId": conversation_id}, {"$set": {"messages": conversation_messages}})

In [ ]:
SYSTEM_PROMT = "you are a helpful assistant"
USER_PROMPT = "Tell me about virat kohli"

messages = [
    {"role": "system", "content": SYSTEM_PROMT},
    {"role": "user", "content": USER_PROMPT},
]

In [ ]:
messages.append({"role": "user", "content": "why we call him as king?"})
call_llm(messages, "12345")

In [ ]:
conversation = conversations.find_one({"userId": "12345", "conversationId": "17a5da20-231c-4d4b-9b04-125fb8f36bb0"})

if conversation and not messages:
    messages = conversation["messages"]
print(messages)

In [ ]:
messages.append({"role": "user", "content": "what is the name of his wife?"})
call_llm(messages, "12345")